In [1]:

from pathlib import Path
import pandas as pd
import json

# Project root
ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

# Processed dataset paths
TRAIN_PATH = (
    ROOT / "data" / "processed" / "banking77" / "train.csv"
)

TEST_PATH = (
    ROOT / "data" / "processed" / "banking77" / "test.csv"
)

CATEGORY_PATH = (
    ROOT / "data" / "processed" / "banking77" / "categories.json"
)

# Verify files
for path in [TRAIN_PATH, TEST_PATH, CATEGORY_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

# Load datasets
banking_train_df = pd.read_csv(TRAIN_PATH)
banking_test_df = pd.read_csv(TEST_PATH)

# Actual BANKING77 schema uses text and category
REQUIRED_COLUMNS = {"text", "category"}

for name, df in [
    ("Training", banking_train_df),
    ("Testing", banking_test_df),
]:
    if not REQUIRED_COLUMNS.issubset(df.columns):
        raise ValueError(
            f"{name} dataset columns incorrect: {df.columns.tolist()}"
        )

    df["text"] = df["text"].astype(str).str.strip()
    df["category"] = df["category"].astype(str).str.strip()

    if (df["text"] == "").any() or (df["category"] == "").any():
        raise ValueError(f"Empty values found in {name} dataset")

# Basic statistics
print("=== BANKING77 DATASET AUDIT ===")

print("\nTraining shape:", banking_train_df.shape)
print("Testing shape:", banking_test_df.shape)

print("\nTraining unique categories:",
      banking_train_df["category"].nunique())

print("Testing unique categories:",
      banking_test_df["category"].nunique())

print("\nTraining columns:", banking_train_df.columns.tolist())
print("Testing columns:", banking_test_df.columns.tolist())

# Class coverage
train_categories = set(banking_train_df["category"].unique())
test_categories = set(banking_test_df["category"].unique())

print("\nCategories missing from training:",
      sorted(test_categories - train_categories))

print("Categories missing from testing:",
      sorted(train_categories - test_categories))

# Exact normalized text overlap
train_texts = set(
    banking_train_df["text"].str.lower().str.strip()
)

test_texts = set(
    banking_test_df["text"].str.lower().str.strip()
)

print("\nExact normalized query overlap:",
      len(train_texts & test_texts))

# Duplicate check
print("\nTraining duplicate rows:",
      banking_train_df.duplicated().sum())

print("Testing duplicate rows:",
      banking_test_df.duplicated().sum())

# Training label distribution
print("\n=== TOP 15 TRAINING CATEGORIES ===")

display(
    banking_train_df["category"]
    .value_counts()
    .head(15)
    .rename_axis("category")
    .reset_index(name="count")
)

# Category metadata
with open(CATEGORY_PATH, "r", encoding="utf-8") as f:
    banking_categories = json.load(f)

print("\nCategory metadata type:", type(banking_categories).__name__)
print("Dataset audit completed.")


=== BANKING77 DATASET AUDIT ===

Training shape: (10003, 2)
Testing shape: (3080, 2)

Training unique categories: 77
Testing unique categories: 77

Training columns: ['text', 'category']
Testing columns: ['text', 'category']

Categories missing from training: []
Categories missing from testing: []

Exact normalized query overlap: 6

Training duplicate rows: 4
Testing duplicate rows: 1

=== TOP 15 TRAINING CATEGORIES ===


,category,count
0,card_payment_fee_charged,187
1,direct_debit_payment_not_recognised,182
2,balance_not_updated_after_cheque_or_cash_deposit,181
3,wrong_amount_of_cash_received,180
4,cash_withdrawal_charge,177
5,transaction_charged_twice,175
6,declined_cash_withdrawal,173
7,transfer_fee_charged,172
8,transfer_not_received_by_recipient,171
9,balance_not_updated_after_bank_transfer,171



Category metadata type: list
Dataset audit completed.


In [2]:
import pandas as pd

# Normalize text for duplicate detection only
train_check = banking_train_df.copy()
test_check = banking_test_df.copy()

train_check["normalized_text"] = (
    train_check["text"].str.lower().str.strip()
)

test_check["normalized_text"] = (
    test_check["text"].str.lower().str.strip()
)

# Find queries present in both official splits
overlap_texts = (
    set(train_check["normalized_text"])
    & set(test_check["normalized_text"])
)

print("=== TRAIN / TEST OVERLAP ANALYSIS ===")
print("Overlapping normalized queries:", len(overlap_texts))

# Display overlapping records and their labels
overlap_details = []

for query in sorted(overlap_texts):
    train_rows = train_check[
        train_check["normalized_text"] == query
    ]

    test_rows = test_check[
        test_check["normalized_text"] == query
    ]

    overlap_details.append({
        "normalized_query": query,
        "train_labels": sorted(train_rows["category"].unique()),
        "test_labels": sorted(test_rows["category"].unique()),
        "train_occurrences": len(train_rows),
        "test_occurrences": len(test_rows),
        "same_label": (
            set(train_rows["category"])
            == set(test_rows["category"])
        ),
    })

overlap_df = pd.DataFrame(overlap_details)

display(overlap_df)

# Check internal duplicate queries
print("\n=== TRAINING DUPLICATES ===")

train_duplicates = (
    train_check[
        train_check.duplicated("normalized_text", keep=False)
    ]
    .sort_values("normalized_text")
)

display(train_duplicates[
    ["text", "category"]
].head(20))

print("Training duplicate records:", 
      int(train_check.duplicated("normalized_text").sum()))

print("\n=== TESTING DUPLICATES ===")

test_duplicates = (
    test_check[
        test_check.duplicated("normalized_text", keep=False)
    ]
    .sort_values("normalized_text")
)

display(test_duplicates[
    ["text", "category"]
])

print("Test duplicate records:",
      int(test_check.duplicated("normalized_text").sum()))

# Category consistency
print("\n=== LABEL CONSISTENCY ===")

if not overlap_df.empty:
    print("Overlapping queries with same labels:",
          int(overlap_df["same_label"].sum()))

    print("Overlapping queries with different labels:",
          int((~overlap_df["same_label"]).sum()))

# Metadata validation
print("\nCategories in metadata:", len(banking_categories))
print("Categories in training:", banking_train_df["category"].nunique())

assert len(banking_categories) == 77
assert banking_train_df["category"].nunique() == 77

print("\nInvestigation complete.")

=== TRAIN / TEST OVERLAP ANALYSIS ===
Overlapping normalized queries: 6


,normalized_query,train_labels,test_labels,train_occurrences,test_occurrences,same_label
0,at which atms can i use this card?,[atm_support],[atm_support],1,1,True
1,how do i unblock my pin?,[pin_blocked],[pin_blocked],1,1,True
2,there are a few transaction that i don't recog...,[compromised_card],[compromised_card],1,1,True
3,what businesses accept this card?,[card_acceptance],[card_acceptance],1,1,True
4,where can i use my card?,[card_acceptance],[card_acceptance],1,1,True
5,which cash machines will allow me to change my...,[change_pin],[change_pin],1,1,True



=== TRAINING DUPLICATES ===


,text,category
6910,Do I need to go to a physical bank to change m...,change_pin
6965,Do I need to go to a physical bank to change m...,change_pin
1246,I can't seem to be able to use my card,card_not_working
1290,I can't seem to be able to use my card,card_not_working
1710,I put the wrong pin too many times and now it ...,pin_blocked
1724,I put the wrong pin too many times and now it ...,pin_blocked
4594,Where can I withdraw money from?,atm_support
4595,Where can I withdraw money from?,atm_support


Training duplicate records: 4

=== TESTING DUPLICATES ===


,text,category
1441,Which ATMs accept this card?,atm_support
1461,Which ATMs accept this card?,atm_support


Test duplicate records: 1

=== LABEL CONSISTENCY ===
Overlapping queries with same labels: 6
Overlapping queries with different labels: 0

Categories in metadata: 77
Categories in training: 77

Investigation complete.


In [3]:
import json
import pandas as pd

# Original datasets ko modify nahi karna
train_clean = banking_train_df.copy()
test_official = banking_test_df.copy()

# Normalize text for duplicate and leakage detection
train_clean["normalized_text"] = (
    train_clean["text"].str.lower().str.strip()
)

test_official["normalized_text"] = (
    test_official["text"].str.lower().str.strip()
)

# Verify that repeated queries have consistent labels
assert (
    train_clean.groupby("normalized_text")["category"].nunique().max() == 1
), "Conflicting labels found in training data!"

assert (
    test_official.groupby("normalized_text")["category"].nunique().max() == 1
), "Conflicting labels found in test data!"

# Identify queries that appear in both official splits
test_query_set = set(test_official["normalized_text"])

overlapping_queries = (
    set(train_clean["normalized_text"]) & test_query_set
)

print("Overlapping queries:", len(overlapping_queries))

# Remove only the overlapping copies from training
before_overlap_removal = len(train_clean)

train_clean = train_clean[
    ~train_clean["normalized_text"].isin(overlapping_queries)
].copy()

overlap_rows_removed = before_overlap_removal - len(train_clean)

# Remove repeated normalized queries within training
before_deduplication = len(train_clean)

train_clean = train_clean.drop_duplicates(
    subset=["normalized_text"],
    keep="first",
).copy()

duplicate_rows_removed = (
    before_deduplication - len(train_clean)
)

# Recheck leakage
remaining_overlap = (
    set(train_clean["normalized_text"])
    & set(test_official["normalized_text"])
)

assert len(remaining_overlap) == 0

# Preserve original official test dataset, including its duplicate
test_duplicate_count = int(
    test_official.duplicated(
        subset=["normalized_text"]
    ).sum()
)

# Remove helper column before saving
train_clean = train_clean.drop(
    columns=["normalized_text"]
)

test_official = test_official.drop(
    columns=["normalized_text"]
)

# Save a separate leakage-safe training file
OUTPUT_DIR = ROOT / "data" / "processed" / "banking77"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_TRAIN_PATH = OUTPUT_DIR / "train_leakage_safe.csv"
CLEAN_REPORT_PATH = (
    ROOT / "reports" / "data_quality"
    / "banking77_leakage_cleaning_report.json"
)

train_clean.to_csv(
    CLEAN_TRAIN_PATH,
    index=False,
    encoding="utf-8",
)

# Cleaning report
cleaning_report = {
    "original_train_rows": int(len(banking_train_df)),
    "overlap_queries_removed_from_train": int(
        overlap_rows_removed
    ),
    "training_duplicate_rows_removed": int(
        duplicate_rows_removed
    ),
    "final_training_rows": int(len(train_clean)),
    "official_test_rows_preserved": int(len(test_official)),
    "duplicates_inside_official_test_preserved": int(
        test_duplicate_count
    ),
    "remaining_train_test_overlap": int(len(remaining_overlap)),
    "training_categories": int(train_clean["category"].nunique()),
    "testing_categories": int(test_official["category"].nunique()),
    "official_test_modified": False,
}

CLEAN_REPORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with CLEAN_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(cleaning_report, f, indent=2)

print("\n=== BANKING77 CLEANING COMPLETE ===")
print(json.dumps(cleaning_report, indent=2))

print("\nCleaned training file:", CLEAN_TRAIN_PATH)
print("Original train file remains unchanged.")
print("Original test file remains unchanged.")

Overlapping queries: 6

=== BANKING77 CLEANING COMPLETE ===
{
  "original_train_rows": 10003,
  "overlap_queries_removed_from_train": 6,
  "training_duplicate_rows_removed": 4,
  "final_training_rows": 9993,
  "official_test_rows_preserved": 3080,
  "duplicates_inside_official_test_preserved": 1,
  "remaining_train_test_overlap": 0,
  "training_categories": 77,
  "testing_categories": 77,
  "official_test_modified": false
}

Cleaned training file: d:\Data scientist\AI Customer Support Intelligence\data\processed\banking77\train_leakage_safe.csv
Original train file remains unchanged.
Original test file remains unchanged.


In [4]:
from sklearn.model_selection import train_test_split

# Load leakage-safe training dataset
banking_train_clean = pd.read_csv(CLEAN_TRAIN_PATH)

# Official test data remains separate and untouched
banking_test_clean = pd.read_csv(TEST_PATH)

# Features and target
X = banking_train_clean["text"].astype(str)
y = banking_train_clean["category"].astype(str)

# Train / Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# Verify splits
print("=== BANKING77 TRAIN / VALIDATION SPLIT ===")

print("\nTraining records:", len(X_train))
print("Validation records:", len(X_val))
print("Official test records:", len(banking_test_clean))

print("\nTraining categories:", y_train.nunique())
print("Validation categories:", y_val.nunique())
print("Test categories:", banking_test_clean["category"].nunique())

# Normalized query overlap check
train_queries = set(X_train.str.lower().str.strip())
val_queries = set(X_val.str.lower().str.strip())
test_queries = set(
    banking_test_clean["text"].str.lower().str.strip()
)

print("\n=== DATA LEAKAGE CHECK ===")
print("Train / Validation overlap:", len(train_queries & val_queries))
print("Train / Test overlap:", len(train_queries & test_queries))
print("Validation / Test overlap:", len(val_queries & test_queries))

# Assertions
assert len(train_queries & val_queries) == 0
assert len(train_queries & test_queries) == 0
assert len(val_queries & test_queries) == 0

assert y_train.nunique() == 77
assert y_val.nunique() == 77

print("\nAll split and leakage checks passed.")

print("\nTraining category distribution:")
display(y_train.value_counts().describe())

print("\nValidation category distribution:")
display(y_val.value_counts().describe())

display(
    pd.DataFrame({
        "Training": y_train.value_counts(),
        "Validation": y_val.value_counts(),
    }).sort_index()
)

=== BANKING77 TRAIN / VALIDATION SPLIT ===

Training records: 7994
Validation records: 1999
Official test records: 3080

Training categories: 77
Validation categories: 77
Test categories: 77

=== DATA LEAKAGE CHECK ===
Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0

All split and leakage checks passed.

Training category distribution:


count     77.000000
mean     103.818182
std       26.413577
min       28.000000
25%       89.000000
50%      102.000000
75%      127.000000
max      149.000000
Name: count, dtype: float64


Validation category distribution:


count    77.000000
mean     25.961039
std       6.673675
min       7.000000
25%      22.000000
50%      25.000000
75%      32.000000
max      38.000000
Name: count, dtype: float64

,Training,Validation
category,,
Refund_not_showing_up,130,32
activate_my_card,127,32
age_limit,88,22
apple_pay_or_google_pay,101,25
atm_support,68,17
...,...,...
virtual_card_not_working,33,8
visa_or_mastercard,108,27
why_verify_identity,97,24


In [5]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF configuration
vectorizer_banking = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    sublinear_tf=True,
    dtype=np.float32,
)

print("Converting training text into TF-IDF features...")

# Fit ONLY on training data
X_train_vec = vectorizer_banking.fit_transform(X_train)

print("Transforming validation text...")

# Validation data uses the same training vocabulary
X_val_vec = vectorizer_banking.transform(X_val)

print("\n=== TF-IDF FEATURE ENGINEERING RESULTS ===")

print("Training records:", X_train_vec.shape[0])
print("Validation records:", X_val_vec.shape[0])

print("Vocabulary size:", len(vectorizer_banking.vocabulary_))

print("\nTraining matrix shape:", X_train_vec.shape)
print("Validation matrix shape:", X_val_vec.shape)

# Calculate sparse matrix density
density = (
    X_train_vec.nnz
    / (X_train_vec.shape[0] * X_train_vec.shape[1])
)

print(f"\nTraining matrix density: {density:.6%}")

# Inspect sample vocabulary
print("\nSample vocabulary terms:")
print(list(vectorizer_banking.vocabulary_.keys())[:30])

# Inspect a few TF-IDF features
feature_names = vectorizer_banking.get_feature_names_out()

feature_sample = pd.DataFrame(
    X_train_vec[:3].toarray(),
    columns=feature_names,
    index=X_train.index[:3],
)

print("\nSample feature matrix:")
display(feature_sample.iloc[:, :15])

print("\nTF-IDF transformation completed successfully.")

Converting training text into TF-IDF features...
Transforming validation text...

=== TF-IDF FEATURE ENGINEERING RESULTS ===
Training records: 7994
Validation records: 1999
Vocabulary size: 8880

Training matrix shape: (7994, 8880)
Validation matrix shape: (1999, 8880)

Training matrix density: 0.207788%

Sample vocabulary terms:
['how', 'do', 'see', 'my', 'updated', 'balance', 'after', 'make', 'cheque', 'or', 'cash', 'deposit', 'how do', 'do see', 'see my', 'updated balance', 'balance after', 'after make', 'cheque or', 'or cash', 'cash deposit', 'what', 'are', 'options', 'for', 'holding', 'money', 'in', 'multiple', 'currencies']

Sample feature matrix:


,00,000,10,10 minutes,10 pounds,100,100 all,100 but,18,18 years,1l,20,20 00,20 came,20 what
3354,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
985,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4811,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



TF-IDF transformation completed successfully.


In [6]:
import time
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    f1_score,
)

# Define models
banking_models = {
    "Logistic Regression": LogisticRegression(
        C=4.0,
        max_iter=2000,
        solver="lbfgs",
        n_jobs=-1,
        random_state=42,
    ),

    "Linear SVM": LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42,
    ),

    "Multinomial Naive Bayes": MultinomialNB(
        alpha=0.3,
    ),
}

# Store trained models and predictions
trained_banking_models = {}
banking_predictions = {}
banking_results = []

print("=== BANKING77 MODEL TRAINING ===")

for model_name, model in banking_models.items():

    print(f"\nTraining: {model_name}")

    start_time = time.time()

    # Train model
    model.fit(X_train_vec, y_train)

    training_time = time.time() - start_time

    # Validation predictions
    prediction_start = time.time()

    y_pred = model.predict(X_val_vec)

    prediction_time = time.time() - prediction_start

    # Evaluation metrics
    accuracy = accuracy_score(y_val, y_pred)

    macro_f1 = f1_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_val,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    # Save results in memory
    trained_banking_models[model_name] = model
    banking_predictions[model_name] = y_pred

    banking_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "Training Time (sec)": training_time,
        "Prediction Time (sec)": prediction_time,
    })

    print(f"Accuracy: {accuracy:.4%}")
    print(f"Macro F1: {macro_f1:.4%}")
    print(f"Weighted F1: {weighted_f1:.4%}")
    print(f"Training Time: {training_time:.2f} sec")
    print(f"Prediction Time: {prediction_time:.4f} sec")

# Final comparison
banking_results_df = pd.DataFrame(banking_results)

banking_results_df = banking_results_df.sort_values(
    by="Macro F1",
    ascending=False,
).reset_index(drop=True)

print("\n=== BANKING77 MODEL COMPARISON ===")

display(
    banking_results_df.style.format({
        "Accuracy": "{:.4%}",
        "Macro F1": "{:.4%}",
        "Weighted F1": "{:.4%}",
        "Training Time (sec)": "{:.2f}",
        "Prediction Time (sec)": "{:.4f}",
    })
)

print("\nBest validation model:",
      banking_results_df.iloc[0]["Model"])

=== BANKING77 MODEL TRAINING ===

Training: Logistic Regression


c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Accuracy: 87.4437%
Macro F1: 87.5713%
Weighted F1: 87.4181%
Training Time: 13.10 sec
Prediction Time: 0.0307 sec

Training: Linear SVM
Accuracy: 87.5938%
Macro F1: 87.6398%
Weighted F1: 87.5562%
Training Time: 0.45 sec
Prediction Time: 0.0030 sec

Training: Multinomial Naive Bayes
Accuracy: 82.0410%
Macro F1: 80.2851%
Weighted F1: 81.6146%
Training Time: 0.09 sec
Prediction Time: 0.0090 sec

=== BANKING77 MODEL COMPARISON ===


,Model,Accuracy,Macro F1,Weighted F1,Training Time (sec),Prediction Time (sec)
0,Linear SVM,87.5938%,87.6398%,87.5562%,0.45,0.0030
1,Logistic Regression,87.4437%,87.5713%,87.4181%,13.10,0.0307
2,Multinomial Naive Bayes,82.0410%,80.2851%,81.6146%,0.09,0.0090



Best validation model: Linear SVM


In [7]:
import numpy as np
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix

# Best model predictions
best_banking_model = trained_banking_models["Linear SVM"]
y_pred_svm = banking_predictions["Linear SVM"]

# Detailed classification report
report = classification_report(
    y_val,
    y_pred_svm,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T

# Keep only actual intent categories
class_report_df = report_df.loc[
    report_df.index.isin(y_train.unique())
].copy()

class_report_df = class_report_df.sort_values(
    by="f1-score",
    ascending=True,
)

print("=== 15 LOWEST PERFORMING INTENTS ===")

display(
    class_report_df[
        ["precision", "recall", "f1-score", "support"]
    ].head(15).round(4)
)

# Overall metrics
print("\n=== OVERALL VALIDATION METRICS ===")

display(
    report_df.loc[
        ["accuracy", "macro avg", "weighted avg"]
    ].round(4)
)

# Confusion matrix
labels = sorted(y_train.unique())

cm = confusion_matrix(
    y_val,
    y_pred_svm,
    labels=labels,
)

# Find strongest incorrect confusion pairs
confusion_pairs = []

for i, actual_label in enumerate(labels):
    for j, predicted_label in enumerate(labels):

        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                "Actual Intent": actual_label,
                "Predicted Intent": predicted_label,
                "Misclassified Samples": cm[i, j],
            })

confusion_df = pd.DataFrame(confusion_pairs)

if not confusion_df.empty:
    confusion_df = confusion_df.sort_values(
        "Misclassified Samples",
        ascending=False,
    )

print("\n=== TOP 15 CONFUSION PAIRS ===")

display(confusion_df.head(15))

# Inspect actual incorrect predictions
error_df = pd.DataFrame({
    "Text": X_val.reset_index(drop=True),
    "Actual Intent": y_val.reset_index(drop=True),
    "Predicted Intent": pd.Series(y_pred_svm),
})

error_df = error_df[
    error_df["Actual Intent"] != error_df["Predicted Intent"]
].reset_index(drop=True)

print("\n=== VALIDATION ERROR SUMMARY ===")
print("Total validation samples:", len(y_val))
print("Incorrect predictions:", len(error_df))
print("Correct predictions:", len(y_val) - len(error_df))
print("Error rate:", f"{len(error_df) / len(y_val):.2%}")

print("\n=== 20 MISCLASSIFIED EXAMPLES ===")
display(error_df.head(20))

=== 15 LOWEST PERFORMING INTENTS ===


,precision,recall,f1-score,support
top_up_failed,0.8261,0.6552,0.7308,29.0
pending_transfer,0.8077,0.7000,0.7500,30.0
top_up_by_bank_transfer_charge,0.6923,0.8182,0.7500,22.0
atm_support,0.9167,0.6471,0.7586,17.0
failed_transfer,0.7097,0.8148,0.7586,27.0
lost_or_stolen_card,0.8000,0.7500,0.7742,16.0
get_disposable_virtual_card,0.8235,0.7368,0.7778,19.0
card_payment_fee_charged,0.7692,0.7895,0.7792,38.0
unable_to_verify_identity,0.7619,0.8000,0.7805,20.0
receiving_money,0.7895,0.7895,0.7895,19.0



=== OVERALL VALIDATION METRICS ===


,precision,recall,f1-score,support
accuracy,0.8759,0.8759,0.8759,0.8759
macro avg,0.8825,0.8749,0.8764,1999.0000
weighted avg,0.8797,0.8759,0.8756,1999.0000



=== TOP 15 CONFUSION PAIRS ===


,Actual Intent,Predicted Intent,Misclassified Samples
147,top_up_failed,pending_top_up,5
96,get_disposable_virtual_card,disposable_card_limits,4
121,pending_transfer,transfer_not_received_by_recipient,4
68,declined_transfer,failed_transfer,3
186,why_verify_identity,unable_to_verify_identity,3
169,transfer_not_received_by_recipient,balance_not_updated_after_bank_transfer,3
120,pending_transfer,pending_card_payment,3
76,disposable_card_limits,get_disposable_virtual_card,3
43,card_payment_not_recognised,direct_debit_payment_not_recognised,3
39,card_payment_fee_charged,transfer_fee_charged,3



=== VALIDATION ERROR SUMMARY ===
Total validation samples: 1999
Incorrect predictions: 248
Correct predictions: 1751
Error rate: 12.41%

=== 20 MISCLASSIFIED EXAMPLES ===


,Text,Actual Intent,Predicted Intent
0,Why does my identity need to be verified?,why_verify_identity,unable_to_verify_identity
1,What is the process for unlocking the pin?,pin_blocked,get_physical_card
2,How is this transfer supposed to work? It's no...,failed_transfer,card_not_working
3,"Hi, I have been transferring money to my frien...",failed_transfer,transfer_fee_charged
4,Will a cheque suffice?,top_up_by_cash_or_cheque,balance_not_updated_after_cheque_or_cash_deposit
5,Do I have to pick a creditcard brand?,visa_or_mastercard,age_limit
6,Why is your exchange rate different on differe...,exchange_rate,card_payment_wrong_exchange_rate
7,I didn't cancel my top-up.,top_up_reverted,top_up_failed
8,What was the cause for my card to be declined?,declined_cash_withdrawal,declined_card_payment
9,How long after I receive my statement can I di...,direct_debit_payment_not_recognised,card_delivery_estimate


In [8]:
from pathlib import Path

# Create report directory
REPORT_DIR = Path("reports/models")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save detailed classification report
class_report_df.to_csv(
    REPORT_DIR / "banking77_svm_classification_report.csv"
)

# Save confusion pairs
confusion_df.to_csv(
    REPORT_DIR / "banking77_svm_confusion_pairs.csv",
    index=False
)

# Save incorrect predictions
error_df.to_csv(
    REPORT_DIR / "banking77_svm_validation_errors.csv",
    index=False
)

# Save overall metrics
report_df.loc[
    ["accuracy", "macro avg", "weighted avg"]
].to_csv(
    REPORT_DIR / "banking77_svm_overall_metrics.csv"
)

# Compact summary only
print("=== EVALUATION REPORTS SAVED ===")
print("Directory:", REPORT_DIR)
print("Classification report: Saved")
print("Confusion pairs:", len(confusion_df), "Saved")
print("Misclassified examples:", len(error_df), "Saved")
print("Overall metrics: Saved")

print("\nBest model: Linear SVM")
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred_svm):.2%}")
print(f"Validation Macro F1: {f1_score(y_val, y_pred_svm, average='macro'):.2%}")

=== EVALUATION REPORTS SAVED ===
Directory: reports\models
Classification report: Saved
Confusion pairs: 195 Saved
Misclassified examples: 248 Saved
Overall metrics: Saved

Best model: Linear SVM
Validation Accuracy: 87.59%
Validation Macro F1: 87.64%


In [9]:
import time
import pandas as pd

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

C_VALUES = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]

svm_tuning_results = []
svm_tuned_models = {}

print("=== LINEAR SVM HYPERPARAMETER TUNING ===")

for c_value in C_VALUES:

    model = LinearSVC(
        C=c_value,
        max_iter=5000,
        random_state=42
    )

    start = time.time()
    model.fit(X_train_vec, y_train)
    training_time = time.time() - start

    predictions = model.predict(X_val_vec)

    accuracy = accuracy_score(y_val, predictions)
    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    svm_tuned_models[c_value] = model

    svm_tuning_results.append({
        "C": c_value,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Training Time (sec)": training_time
    })

# Sort by Macro F1
svm_tuning_df = pd.DataFrame(svm_tuning_results)

svm_tuning_df = svm_tuning_df.sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

# Select best C
best_svm_c = float(svm_tuning_df.iloc[0]["C"])
best_svm_model = svm_tuned_models[best_svm_c]

# Save tuning report
svm_tuning_df.to_csv(
    REPORT_DIR / "banking77_svm_hyperparameter_tuning.csv",
    index=False
)

print("\n=== TUNING SUMMARY ===")

for _, row in svm_tuning_df.iterrows():
    print(
        f"C={row['C']:<4} | "
        f"Accuracy={row['Accuracy']:.2%} | "
        f"Macro F1={row['Macro F1']:.2%}"
    )

print("\nBest C:", best_svm_c)
print(f"Best Macro F1: {svm_tuning_df.iloc[0]['Macro F1']:.2%}")
print("Tuning report saved.")

=== LINEAR SVM HYPERPARAMETER TUNING ===

=== TUNING SUMMARY ===
C=1.0  | Accuracy=87.59% | Macro F1=87.64%
C=2.0  | Accuracy=87.24% | Macro F1=87.32%
C=0.5  | Accuracy=86.99% | Macro F1=87.01%
C=4.0  | Accuracy=86.99% | Macro F1=86.99%
C=0.25 | Accuracy=86.54% | Macro F1=86.53%
C=8.0  | Accuracy=86.54% | Macro F1=86.52%

Best C: 1.0
Best Macro F1: 87.64%
Tuning report saved.


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
from scipy.sparse import hstack
import time
import numpy as np

# Character-level TF-IDF
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=30000,
    sublinear_tf=True,
    dtype=np.float32
)

print("Creating character-level TF-IDF features...")

X_train_char = char_vectorizer.fit_transform(X_train)
X_val_char = char_vectorizer.transform(X_val)

# Combine word and character features
CHAR_WEIGHT = 0.7

X_train_combined = hstack([
    X_train_vec,
    X_train_char.multiply(CHAR_WEIGHT)
]).tocsr()

X_val_combined = hstack([
    X_val_vec,
    X_val_char.multiply(CHAR_WEIGHT)
]).tocsr()

print("Word features:", X_train_vec.shape[1])
print("Character features:", X_train_char.shape[1])
print("Combined features:", X_train_combined.shape[1])

# Train combined-feature model
combined_svm = LinearSVC(
    C=1.0,
    max_iter=5000,
    random_state=42
)

start = time.time()

combined_svm.fit(X_train_combined, y_train)

training_time = time.time() - start

# Validation
combined_predictions = combined_svm.predict(X_val_combined)

combined_accuracy = accuracy_score(
    y_val, combined_predictions
)

combined_macro_f1 = f1_score(
    y_val,
    combined_predictions,
    average="macro"
)

# Compare with previous word-only model
baseline_macro_f1 = f1_score(
    y_val,
    banking_predictions["Linear SVM"],
    average="macro"
)

print("\n=== WORD vs WORD + CHARACTER RESULTS ===")
print(f"Word-only Macro F1: {baseline_macro_f1:.4%}")
print(f"Word + Character Accuracy: {combined_accuracy:.4%}")
print(f"Word + Character Macro F1: {combined_macro_f1:.4%}")
print(f"Training Time: {training_time:.2f} sec")

print(
    "Improvement:",
    f"{(combined_macro_f1 - baseline_macro_f1) * 100:.3f}",
    "percentage points"
)

# Store candidate model
if combined_macro_f1 > baseline_macro_f1:
    print("\nCombined features improved validation performance.")
else:
    print("\nWord-only model remains the better candidate.")

Creating character-level TF-IDF features...
Word features: 8880
Character features: 12691
Combined features: 21571

=== WORD vs WORD + CHARACTER RESULTS ===
Word-only Macro F1: 87.6398%
Word + Character Accuracy: 89.3947%
Word + Character Macro F1: 89.4592%
Training Time: 3.54 sec
Improvement: 1.819 percentage points

Combined features improved validation performance.


In [11]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# Combined model classification report
combined_report = classification_report(
    y_val,
    combined_predictions,
    output_dict=True,
    zero_division=0
)

combined_report_df = pd.DataFrame(combined_report).T

# Extract intent-level metrics
combined_class_report = combined_report_df.loc[
    combined_report_df.index.isin(y_train.unique())
].copy()

combined_class_report = combined_class_report.sort_values(
    "f1-score",
    ascending=True
)

# Find lowest performing categories
lowest_10 = combined_class_report[
    ["precision", "recall", "f1-score", "support"]
].head(10)

# Confusion matrix
labels = sorted(y_train.unique())

combined_cm = confusion_matrix(
    y_val,
    combined_predictions,
    labels=labels
)

# Find incorrect class pairs
confusion_pairs_combined = []

for i, actual in enumerate(labels):
    for j, predicted in enumerate(labels):
        if i != j and combined_cm[i, j] > 0:
            confusion_pairs_combined.append({
                "Actual": actual,
                "Predicted": predicted,
                "Count": combined_cm[i, j]
            })

confusion_pairs_combined = pd.DataFrame(
    confusion_pairs_combined
)

if not confusion_pairs_combined.empty:
    confusion_pairs_combined = (
        confusion_pairs_combined
        .sort_values("Count", ascending=False)
    )

# Save complete reports
combined_class_report.to_csv(
    REPORT_DIR / "banking77_combined_svm_classification_report.csv"
)

confusion_pairs_combined.to_csv(
    REPORT_DIR / "banking77_combined_svm_confusion_pairs.csv",
    index=False
)

print("=== COMBINED SVM EVALUATION ===")
print(f"Accuracy: {accuracy_score(y_val, combined_predictions):.2%}")
print(f"Macro F1: {f1_score(y_val, combined_predictions, average='macro'):.2%}")
print("Incorrect predictions:", (combined_predictions != y_val.to_numpy()).sum())

print("\n=== 10 LOWEST PERFORMING INTENTS ===")
display(lowest_10.round(3))

print("\n=== TOP 10 CONFUSION PAIRS ===")
display(confusion_pairs_combined.head(10))

print("\nDetailed reports saved successfully.")

=== COMBINED SVM EVALUATION ===
Accuracy: 89.39%
Macro F1: 89.46%
Incorrect predictions: 212

=== 10 LOWEST PERFORMING INTENTS ===


,precision,recall,f1-score,support
pending_transfer,0.792,0.633,0.704,30.0
top_up_failed,0.864,0.655,0.745,29.0
compromised_card,0.714,0.882,0.789,17.0
lost_or_stolen_card,0.857,0.750,0.800,16.0
top_up_by_bank_transfer_charge,0.783,0.818,0.800,22.0
why_verify_identity,0.857,0.750,0.800,24.0
transfer_not_received_by_recipient,0.818,0.794,0.806,34.0
pending_top_up,0.781,0.833,0.806,30.0
get_disposable_virtual_card,0.800,0.842,0.821,19.0
transfer_timing,0.840,0.808,0.824,26.0



=== TOP 10 CONFUSION PAIRS ===


,Actual,Predicted,Count
66,disposable_card_limits,get_disposable_virtual_card,4
127,top_up_failed,pending_top_up,4
102,pending_transfer,pending_card_payment,4
104,pending_transfer,transfer_not_received_by_recipient,4
147,transfer_not_received_by_recipient,balance_not_updated_after_bank_transfer,3
38,card_payment_fee_charged,transfer_fee_charged,3
63,direct_debit_payment_not_recognised,card_payment_not_recognised,3
161,why_verify_identity,unable_to_verify_identity,3
17,beneficiary_not_allowed,failed_transfer,3
42,card_payment_not_recognised,verify_source_of_funds,2



Detailed reports saved successfully.


In [12]:
import time
import pandas as pd

from scipy.sparse import hstack
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

CHAR_WEIGHTS = [0.3, 0.5, 0.7, 1.0]

char_weight_results = []
char_weight_models = {}

print("=== CHARACTER FEATURE WEIGHT TUNING ===")

for weight in CHAR_WEIGHTS:

    X_train_weighted = hstack([
        X_train_vec,
        X_train_char.multiply(weight)
    ]).tocsr()

    X_val_weighted = hstack([
        X_val_vec,
        X_val_char.multiply(weight)
    ]).tocsr()

    model = LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    )

    start = time.time()
    model.fit(X_train_weighted, y_train)
    training_time = time.time() - start

    predictions = model.predict(X_val_weighted)

    accuracy = accuracy_score(y_val, predictions)
    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro"
    )

    char_weight_models[weight] = model

    char_weight_results.append({
        "Character Weight": weight,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Training Time (sec)": training_time
    })

# Sort results
char_weight_df = pd.DataFrame(char_weight_results)

char_weight_df = char_weight_df.sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

# Select best weight
best_char_weight = float(
    char_weight_df.iloc[0]["Character Weight"]
)

best_char_model = char_weight_models[best_char_weight]

# Save report
char_weight_df.to_csv(
    REPORT_DIR / "banking77_character_weight_tuning.csv",
    index=False
)

print("\n=== COMPACT RESULTS ===")

for _, row in char_weight_df.iterrows():
    print(
        f"Weight={row['Character Weight']:.1f} | "
        f"Accuracy={row['Accuracy']:.2%} | "
        f"Macro F1={row['Macro F1']:.2%}"
    )

print("\nBest character weight:", best_char_weight)
print(f"Best Macro F1: {char_weight_df.iloc[0]['Macro F1']:.2%}")
print("Report saved successfully.")

=== CHARACTER FEATURE WEIGHT TUNING ===

=== COMPACT RESULTS ===
Weight=1.0 | Accuracy=89.54% | Macro F1=89.59%
Weight=0.7 | Accuracy=89.39% | Macro F1=89.46%
Weight=0.5 | Accuracy=89.14% | Macro F1=89.20%
Weight=0.3 | Accuracy=88.54% | Macro F1=88.56%

Best character weight: 1.0
Best Macro F1: 89.59%
Report saved successfully.


In [13]:
import time
import pandas as pd

from scipy.sparse import hstack
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

# Best character weight from previous experiment
best_char_weight = 1.0

# Prepare combined feature matrices
X_train_best = hstack([
    X_train_vec,
    X_train_char.multiply(best_char_weight)
]).tocsr()

X_val_best = hstack([
    X_val_vec,
    X_val_char.multiply(best_char_weight)
]).tocsr()

C_VALUES = [0.25, 0.5, 1.0, 2.0, 4.0]

combined_c_results = []
combined_c_models = {}

print("=== COMBINED SVM C TUNING ===")

for c_value in C_VALUES:

    model = LinearSVC(
        C=c_value,
        max_iter=5000,
        random_state=42
    )

    start = time.time()
    model.fit(X_train_best, y_train)
    training_time = time.time() - start

    predictions = model.predict(X_val_best)

    accuracy = accuracy_score(y_val, predictions)
    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    combined_c_models[c_value] = model

    combined_c_results.append({
        "C": c_value,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Training Time (sec)": training_time
    })

# Sort by Macro F1
combined_c_df = pd.DataFrame(combined_c_results)

combined_c_df = combined_c_df.sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

# Select best C
best_combined_c = float(combined_c_df.iloc[0]["C"])
best_combined_svm = combined_c_models[best_combined_c]

# Save tuning report
combined_c_df.to_csv(
    REPORT_DIR / "banking77_combined_svm_c_tuning.csv",
    index=False
)

print("\n=== COMPACT RESULTS ===")

for _, row in combined_c_df.iterrows():
    print(
        f"C={row['C']:.2f} | "
        f"Accuracy={row['Accuracy']:.2%} | "
        f"Macro F1={row['Macro F1']:.2%}"
    )

print("\nBest C:", best_combined_c)
print(f"Best Macro F1: {combined_c_df.iloc[0]['Macro F1']:.2%}")
print("Tuning report saved.")

=== COMBINED SVM C TUNING ===

=== COMPACT RESULTS ===
C=2.00 | Accuracy=89.49% | Macro F1=89.59%
C=1.00 | Accuracy=89.54% | Macro F1=89.59%
C=0.50 | Accuracy=89.34% | Macro F1=89.34%
C=4.00 | Accuracy=89.09% | Macro F1=89.09%
C=0.25 | Accuracy=88.94% | Macro F1=88.84%

Best C: 2.0
Best Macro F1: 89.59%
Tuning report saved.


In [14]:
import time
import pandas as pd

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

class_weight_results = []
class_weight_models = {}

experiments = {
    "No Class Weight": None,
    "Balanced Class Weight": "balanced"
}

print("=== SVM CLASS WEIGHT EXPERIMENT ===")

for experiment_name, weight_option in experiments.items():

    model = LinearSVC(
        C=1.0,
        class_weight=weight_option,
        max_iter=5000,
        random_state=42
    )

    start = time.time()
    model.fit(X_train_best, y_train)
    training_time = time.time() - start

    predictions = model.predict(X_val_best)

    accuracy = accuracy_score(y_val, predictions)

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    class_weight_models[experiment_name] = model

    class_weight_results.append({
        "Experiment": experiment_name,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Training Time (sec)": training_time
    })

class_weight_df = pd.DataFrame(class_weight_results)

class_weight_df.to_csv(
    REPORT_DIR / "banking77_svm_class_weight_comparison.csv",
    index=False
)

print("\n=== COMPACT RESULTS ===")

for _, row in class_weight_df.iterrows():
    print(
        f"{row['Experiment']} | "
        f"Accuracy={row['Accuracy']:.2%} | "
        f"Macro F1={row['Macro F1']:.2%}"
    )

print("\nReport saved successfully.")

=== SVM CLASS WEIGHT EXPERIMENT ===

=== COMPACT RESULTS ===
No Class Weight | Accuracy=89.54% | Macro F1=89.59%
Balanced Class Weight | Accuracy=89.44% | Macro F1=89.40%

Report saved successfully.


In [15]:
import time
import pandas as pd

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

class_weight_results = []
class_weight_models = {}

experiments = {
    "No Class Weight": None,
    "Balanced Class Weight": "balanced"
}

print("=== SVM CLASS WEIGHT EXPERIMENT ===")

for experiment_name, weight_option in experiments.items():

    model = LinearSVC(
        C=1.0,
        class_weight=weight_option,
        max_iter=5000,
        random_state=42
    )

    start = time.time()
    model.fit(X_train_best, y_train)
    training_time = time.time() - start

    predictions = model.predict(X_val_best)

    accuracy = accuracy_score(y_val, predictions)

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    class_weight_models[experiment_name] = model

    class_weight_results.append({
        "Experiment": experiment_name,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Training Time (sec)": training_time
    })

class_weight_df = pd.DataFrame(class_weight_results)

class_weight_df.to_csv(
    REPORT_DIR / "banking77_svm_class_weight_comparison.csv",
    index=False
)

print("\n=== COMPACT RESULTS ===")

for _, row in class_weight_df.iterrows():
    print(
        f"{row['Experiment']} | "
        f"Accuracy={row['Accuracy']:.2%} | "
        f"Macro F1={row['Macro F1']:.2%}"
    )

print("\nReport saved successfully.")

=== SVM CLASS WEIGHT EXPERIMENT ===

=== COMPACT RESULTS ===
No Class Weight | Accuracy=89.54% | Macro F1=89.59%
Balanced Class Weight | Accuracy=89.44% | Macro F1=89.40%

Report saved successfully.


In [16]:
import json
import time
import joblib
import sklearn
import pandas as pd
import numpy as np

from pathlib import Path
from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# --------------------------------------------------
# 1. Load complete leakage-safe training data
# --------------------------------------------------

final_train_df = pd.read_csv(CLEAN_TRAIN_PATH)
final_test_df = pd.read_csv(TEST_PATH)

X_full = final_train_df["text"].astype(str)
y_full = final_train_df["category"].astype(str)

X_test_final = final_test_df["text"].astype(str)
y_test_final = final_test_df["category"].astype(str)

# Verify no normalized query overlap
normalize = lambda s: s.str.lower().str.strip()

assert len(
    set(normalize(X_full)) & set(normalize(X_test_final))
) == 0

assert y_full.nunique() == 77
assert y_test_final.nunique() == 77

print("Training records:", len(X_full))
print("Official test records:", len(X_test_final))
print("Train-test overlap: 0")

# --------------------------------------------------
# 2. Fit final Word TF-IDF
# --------------------------------------------------

word_vectorizer_final = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_full_word = word_vectorizer_final.fit_transform(X_full)
X_test_word = word_vectorizer_final.transform(X_test_final)

# --------------------------------------------------
# 3. Fit final Character TF-IDF
# --------------------------------------------------

char_vectorizer_final = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=30000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_full_char = char_vectorizer_final.fit_transform(X_full)
X_test_char = char_vectorizer_final.transform(X_test_final)

# Character weight selected from validation experiments
CHAR_WEIGHT = 1.0

X_full_final = hstack([
    X_full_word,
    X_full_char.multiply(CHAR_WEIGHT)
]).tocsr()

X_test_final_vec = hstack([
    X_test_word,
    X_test_char.multiply(CHAR_WEIGHT)
]).tocsr()

# --------------------------------------------------
# 4. Train final Linear SVM
# --------------------------------------------------

final_banking_model = LinearSVC(
    C=1.0,
    class_weight=None,
    max_iter=5000,
    random_state=42,
)

print("\nTraining final model on all safe training data...")

start = time.time()

final_banking_model.fit(X_full_final, y_full)

final_training_time = time.time() - start

# --------------------------------------------------
# 5. Official test evaluation
# --------------------------------------------------

test_predictions = final_banking_model.predict(
    X_test_final_vec
)

test_accuracy = accuracy_score(
    y_test_final,
    test_predictions
)

test_macro_f1 = f1_score(
    y_test_final,
    test_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    y_test_final,
    test_predictions,
    average="weighted",
    zero_division=0
)

# --------------------------------------------------
# 6. Save reports
# --------------------------------------------------

MODEL_DIR = Path("models/intent_classification")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

classification_rep = classification_report(
    y_test_final,
    test_predictions,
    output_dict=True,
    zero_division=0
)

classification_df = pd.DataFrame(classification_rep).T

classification_df.to_csv(
    REPORT_DIR / "banking77_final_test_classification_report.csv"
)

test_errors = pd.DataFrame({
    "Text": X_test_final.reset_index(drop=True),
    "Actual": y_test_final.reset_index(drop=True),
    "Predicted": test_predictions,
})

test_errors = test_errors[
    test_errors["Actual"] != test_errors["Predicted"]
].reset_index(drop=True)

test_errors.to_csv(
    REPORT_DIR / "banking77_final_test_errors.csv",
    index=False
)

labels = sorted(y_full.unique())

cm = confusion_matrix(
    y_test_final,
    test_predictions,
    labels=labels
)

pd.DataFrame(
    cm,
    index=labels,
    columns=labels
).to_csv(
    REPORT_DIR / "banking77_final_test_confusion_matrix.csv"
)

# --------------------------------------------------
# 7. Save final model artifact
# --------------------------------------------------

final_artifact = {
    "model": final_banking_model,
    "word_vectorizer": word_vectorizer_final,
    "char_vectorizer": char_vectorizer_final,
    "char_weight": CHAR_WEIGHT,
    "classes": final_banking_model.classes_,
    "config": {
        "algorithm": "LinearSVC",
        "C": 1.0,
        "class_weight": None,
        "word_ngram_range": [1, 2],
        "char_ngram_range": [3, 5],
        "char_weight": CHAR_WEIGHT,
    },
}

model_path = MODEL_DIR / "banking77_intent_model_final.joblib"

joblib.dump(final_artifact, model_path)

# Save evaluation metadata
final_metrics = {
    "dataset": "BANKING77",
    "training_records": int(len(X_full)),
    "official_test_records": int(len(X_test_final)),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "training_time_seconds": float(final_training_time),
    "word_features": int(X_full_word.shape[1]),
    "character_features": int(X_full_char.shape[1]),
    "combined_features": int(X_full_final.shape[1]),
    "incorrect_predictions": int(len(test_errors)),
    "sklearn_version": sklearn.__version__,
    "test_duplicate_rows_preserved": 1,
}

metrics_path = REPORT_DIR / "banking77_final_test_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=4)

# --------------------------------------------------
# 8. Compact output
# --------------------------------------------------

print("\n=== BANKING77 FINAL OFFICIAL TEST RESULTS ===")

print(f"Accuracy: {test_accuracy:.2%}")
print(f"Macro F1: {test_macro_f1:.2%}")
print(f"Weighted F1: {test_weighted_f1:.2%}")
print("Incorrect predictions:", len(test_errors))
print(f"Training time: {final_training_time:.2f} seconds")

print("\n=== FINAL ARTIFACTS ===")
print("Model:", model_path)
print("Metrics:", metrics_path)
print("Test error report: Saved")
print("Confusion matrix: Saved")
print("Classification report: Saved")

print("\nFinal model training and test evaluation completed.")

Training records: 9993
Official test records: 3080
Train-test overlap: 0

Training final model on all safe training data...

=== BANKING77 FINAL OFFICIAL TEST RESULTS ===
Accuracy: 90.88%
Macro F1: 90.88%
Weighted F1: 90.88%
Incorrect predictions: 281
Training time: 5.20 seconds

=== FINAL ARTIFACTS ===
Model: models\intent_classification\banking77_intent_model_final.joblib
Metrics: reports\models\banking77_final_test_metrics.json
Test error report: Saved
Confusion matrix: Saved
Classification report: Saved

Final model training and test evaluation completed.


In [17]:
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack
from pathlib import Path

# Load final saved model
artifact = joblib.load(
    MODEL_DIR / "banking77_intent_model_final.joblib"
)

model = artifact["model"]
word_vectorizer = artifact["word_vectorizer"]
char_vectorizer = artifact["char_vectorizer"]
char_weight = artifact["char_weight"]

# Verify model configuration
assert len(model.classes_) == 77
assert len(word_vectorizer.vocabulary_) > 0
assert len(char_vectorizer.vocabulary_) > 0

# Prediction function
def predict_banking77_intent(message):

    if not isinstance(message, str) or not message.strip():
        raise ValueError("Message must be a non-empty string.")

    messages = [message.strip()]

    word_features = word_vectorizer.transform(messages)
    char_features = char_vectorizer.transform(messages)

    combined_features = hstack([
        word_features,
        char_features.multiply(char_weight)
    ]).tocsr()

    prediction = model.predict(combined_features)[0]

    # Linear SVM decision scores, NOT probabilities
    scores = model.decision_function(combined_features)[0]

    top_indices = np.argsort(scores)[::-1][:3]

    top_candidates = [
        {
            "intent": str(model.classes_[i]),
            "decision_score": float(scores[i])
        }
        for i in top_indices
    ]

    return {
        "intent": str(prediction),
        "top_candidates": top_candidates
    }


# Test sample messages
test_messages = [
    "My card has been stolen",
    "I am waiting for my bank transfer",
    "Why was I charged an extra fee?",
    "I cannot verify my identity",
    "How do I create a disposable virtual card?"
]

smoke_results = []

for message in test_messages:

    result = predict_banking77_intent(message)

    smoke_results.append({
        "Message": message,
        "Predicted Intent": result["intent"],
        "Top Candidate": result["top_candidates"][0]["intent"]
    })

# Display compact results
smoke_df = pd.DataFrame(smoke_results)

display(smoke_df)

# Save smoke test
smoke_df.to_csv(
    REPORT_DIR / "banking77_model_smoke_test.csv",
    index=False
)

print("\n=== MODEL SMOKE TEST ===")
print("Artifact loaded successfully.")
print("Classes:", len(model.classes_))
print("Test messages:", len(smoke_df))
print("All predictions generated:", smoke_df["Predicted Intent"].notna().all())
print("Smoke test report saved.")

,Message,Predicted Intent,Top Candidate
0,My card has been stolen,lost_or_stolen_card,lost_or_stolen_card
1,I am waiting for my bank transfer,balance_not_updated_after_bank_transfer,balance_not_updated_after_bank_transfer
2,Why was I charged an extra fee?,extra_charge_on_statement,extra_charge_on_statement
3,I cannot verify my identity,why_verify_identity,why_verify_identity
4,How do I create a disposable virtual card?,get_disposable_virtual_card,get_disposable_virtual_card



=== MODEL SMOKE TEST ===
Artifact loaded successfully.
Classes: 77
Test messages: 5
All predictions generated: True
Smoke test report saved.


In [18]:
import torch
import transformers
import pandas as pd

from sklearn.model_selection import train_test_split

# Load leakage-safe training data only
bert_df = pd.read_csv(CLEAN_TRAIN_PATH)

X_bert = bert_df["text"].astype(str)
y_bert = bert_df["category"].astype(str)

# Fresh stratified split for Transformer
X_train_bert, X_val_bert, y_train_bert, y_val_bert = (
    train_test_split(
        X_bert,
        y_bert,
        test_size=0.10,
        random_state=42,
        stratify=y_bert
    )
)

print("=== BANKING77 TRANSFORMER READINESS ===")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("Device: CPU" if not torch.cuda.is_available() else "Device: CUDA")

print("\nTraining records:", len(X_train_bert))
print("Validation records:", len(X_val_bert))

print("Training categories:", y_train_bert.nunique())
print("Validation categories:", y_val_bert.nunique())

# Verify all categories exist
assert y_train_bert.nunique() == 77
assert y_val_bert.nunique() == 77

# Check split leakage
train_set = set(X_train_bert.str.lower().str.strip())
val_set = set(X_val_bert.str.lower().str.strip())

assert len(train_set & val_set) == 0

print("\nTrain-validation overlap:", len(train_set & val_set))
print("All readiness checks passed.")

=== BANKING77 TRANSFORMER READINESS ===
PyTorch: 2.14.1+cpu
Transformers: 5.20.2
Device: CPU

Training records: 8993
Validation records: 1000
Training categories: 77
Validation categories: 77

Train-validation overlap: 0
All readiness checks passed.


In [19]:
import torch
import numpy as np

from transformers import BertTokenizerLegacy

# Reproducibility
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# Model configuration
BERT_MODEL_NAME = "prajjwal1/bert-tiny"
MAX_LENGTH = 96

# Load tokenizer
print("Loading BERT-Tiny tokenizer...")

bert_tokenizer = BertTokenizerLegacy.from_pretrained(
    BERT_MODEL_NAME
)

# Create label mappings
bert_classes = sorted(y_train_bert.unique())

label2id = {
    label: index
    for index, label in enumerate(bert_classes)
}

id2label = {
    index: label
    for label, index in label2id.items()
}

# Convert labels to integer IDs
y_train_ids = torch.tensor(
    y_train_bert.map(label2id).values,
    dtype=torch.long
)

y_val_ids = torch.tensor(
    y_val_bert.map(label2id).values,
    dtype=torch.long
)

# Tokenize training data
print("\nTokenizing training data...")

train_encodings = bert_tokenizer(
    X_train_bert.tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

# Tokenize validation data
print("Tokenizing validation data...")

val_encodings = bert_tokenizer(
    X_val_bert.tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

# Verify preparation
assert train_encodings["input_ids"].shape == (
    len(X_train_bert), MAX_LENGTH
)

assert val_encodings["input_ids"].shape == (
    len(X_val_bert), MAX_LENGTH
)

assert len(label2id) == 77

print("\n=== BERT DATA PREPARATION RESULTS ===")
print("Tokenizer vocabulary:", len(bert_tokenizer))
print("Maximum sequence length:", MAX_LENGTH)
print("Training tensor:", tuple(train_encodings["input_ids"].shape))
print("Validation tensor:", tuple(val_encodings["input_ids"].shape))
print("Intent classes:", len(label2id))
print("Label encoding: Successful")
print("Tokenization: Successful")

Loading BERT-Tiny tokenizer...

Tokenizing training data...
Tokenizing validation data...

=== BERT DATA PREPARATION RESULTS ===
Tokenizer vocabulary: 30522
Maximum sequence length: 96
Training tensor: (8993, 96)
Validation tensor: (1000, 96)
Intent classes: 77
Label encoding: Successful
Tokenization: Successful


In [20]:
import torch

from transformers import BertForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader

# CPU configuration
device = torch.device("cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))

BATCH_SIZE = 16
LEARNING_RATE = 2e-5

# Load pretrained BERT-Tiny with 77-class classification head
print("Loading pretrained BERT-Tiny classification model...")

bert_model = BertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=77,
    id2label=id2label,
    label2id=label2id,
    problem_type="single_label_classification",
)

bert_model.to(device)

# Prepare training dataset
train_dataset = TensorDataset(
    train_encodings["input_ids"],
    train_encodings["attention_mask"],
    y_train_ids,
)

# Prepare validation dataset
val_dataset = TensorDataset(
    val_encodings["input_ids"],
    val_encodings["attention_mask"],
    y_val_ids,
)

# Data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

# Count model parameters
total_parameters = sum(
    parameter.numel()
    for parameter in bert_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in bert_model.parameters()
    if parameter.requires_grad
)

print("\n=== BERT MODEL INITIALIZATION ===")
print("Architecture: BERT-Tiny")
print("Device:", device)
print("Intent classes:", bert_model.config.num_labels)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Model initialization successful.")

Loading pretrained BERT-Tiny classification model...


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 


=== BERT MODEL INITIALIZATION ===
Architecture: BERT-Tiny
Device: cpu
Intent classes: 77
Batch size: 16
Learning rate: 2e-05
Total parameters: 4,395,853
Trainable parameters: 4,395,853
Training batches: 563
Validation batches: 63
Model initialization successful.


In [21]:
import time
import copy
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Training configuration
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# Optimizer
optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Linear learning rate scheduler
total_steps = EPOCHS * len(train_loader)
warmup_steps = int(total_steps * 0.1)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / max(1, warmup_steps)

    return max(
        0.0,
        float(total_steps - current_step)
        / max(1, total_steps - warmup_steps)
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

# History and best checkpoint
bert_training_history = []
best_bert_macro_f1 = -1.0
best_bert_state = None

print("=== BERT-TINY TRAINING STARTED ===")

for epoch in range(EPOCHS):

    epoch_start = time.time()

    # -------------------------------
    # Training phase
    # -------------------------------
    bert_model.train()

    total_train_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress:

        input_ids, attention_mask, labels = [
            tensor.to(device) for tensor in batch
        ]

        optimizer.zero_grad(set_to_none=True)

        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            bert_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_train_loss = (
        total_train_loss / len(train_loader)
    )

    # -------------------------------
    # Validation phase
    # -------------------------------
    bert_model.eval()

    total_val_loss = 0.0
    all_predictions = []
    all_true_labels = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids, attention_mask, labels = [
                tensor.to(device) for tensor in batch
            ]

            outputs = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            loss = torch.nn.functional.cross_entropy(
                outputs.logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_true_labels.extend(
                labels.cpu().numpy()
            )

    average_val_loss = total_val_loss / len(val_loader)

    val_accuracy = accuracy_score(
        all_true_labels,
        all_predictions
    )

    val_macro_f1 = f1_score(
        all_true_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    val_weighted_f1 = f1_score(
        all_true_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    epoch_time = time.time() - epoch_start

    # Save best checkpoint in memory
    if val_macro_f1 > best_bert_macro_f1:

        best_bert_macro_f1 = val_macro_f1

        best_bert_state = {
            key: value.detach().cpu().clone()
            for key, value in bert_model.state_dict().items()
        }

        best_bert_epoch = epoch + 1

    bert_training_history.append({
        "Epoch": epoch + 1,
        "Train Loss": average_train_loss,
        "Validation Loss": average_val_loss,
        "Accuracy": val_accuracy,
        "Macro F1": val_macro_f1,
        "Weighted F1": val_weighted_f1,
        "Time (sec)": epoch_time
    })

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {average_train_loss:.4f}")
    print(f"Validation Loss: {average_val_loss:.4f}")
    print(f"Accuracy: {val_accuracy:.2%}")
    print(f"Macro F1: {val_macro_f1:.2%}")
    print(f"Weighted F1: {val_weighted_f1:.2%}")
    print(f"Time: {epoch_time:.2f} sec")

# Restore best validation checkpoint
bert_model.load_state_dict(best_bert_state)
bert_model.to(device)

bert_history_df = pd.DataFrame(
    bert_training_history
)

print("\n=== BERT TRAINING SUMMARY ===")
print("Completed epochs:", len(bert_training_history))
print("Best epoch:", best_bert_epoch)
print(f"Best Validation Macro F1: {best_bert_macro_f1:.2%}")
print("\nTraining completed successfully.")

=== BERT-TINY TRAINING STARTED ===


Epoch 1/3:   0%|          | 0/563 [00:00<?, ?it/s]


Epoch 1/3
Train Loss: 4.3168
Validation Loss: 4.2333
Accuracy: 8.60%
Macro F1: 4.62%
Weighted F1: 5.11%
Time: 129.12 sec


Epoch 2/3:   0%|          | 0/563 [00:00<?, ?it/s]


Epoch 2/3
Train Loss: 4.1971
Validation Loss: 4.1000
Accuracy: 20.40%
Macro F1: 14.29%
Weighted F1: 15.15%
Time: 87.45 sec


Epoch 3/3:   0%|          | 0/563 [00:00<?, ?it/s]


Epoch 3/3
Train Loss: 4.1141
Validation Loss: 4.0512
Accuracy: 22.70%
Macro F1: 15.74%
Weighted F1: 16.63%
Time: 85.97 sec

=== BERT TRAINING SUMMARY ===
Completed epochs: 3
Best epoch: 3
Best Validation Macro F1: 15.74%

Training completed successfully.


In [22]:
import torch
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, f1_score

# Evaluation helper
def evaluate_bert_loader(loader):

    bert_model.eval()

    all_predictions = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():

        for input_ids, attention_mask, labels in loader:

            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            loss = torch.nn.functional.cross_entropy(
                outputs.logits,
                labels
            )

            predictions = outputs.logits.argmax(dim=1)

            total_loss += loss.item()

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    return {
        "loss": total_loss / len(loader),
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_f1": f1_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        ),
        "predictions": np.array(all_predictions),
        "labels": np.array(all_labels),
    }


# Sequential training loader for accurate diagnostics
train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Running training diagnostics...")

train_diag = evaluate_bert_loader(train_eval_loader)
val_diag = evaluate_bert_loader(val_loader)

# Prediction diversity
train_unique_predictions = len(
    np.unique(train_diag["predictions"])
)

val_unique_predictions = len(
    np.unique(val_diag["predictions"])
)

# Check label mappings
mapping_valid = (
    len(label2id) == 77
    and len(id2label) == 77
    and len(np.unique(y_train_ids.numpy())) == 77
    and len(np.unique(y_val_ids.numpy())) == 77
)

print("\n=== BERT TRAINING DIAGNOSTIC ===")

print(f"Training Accuracy: {train_diag['accuracy']:.2%}")
print(f"Training Macro F1: {train_diag['macro_f1']:.2%}")
print(f"Training Loss: {train_diag['loss']:.4f}")

print(f"\nValidation Accuracy: {val_diag['accuracy']:.2%}")
print(f"Validation Macro F1: {val_diag['macro_f1']:.2%}")
print(f"Validation Loss: {val_diag['loss']:.4f}")

print("\nTraining predicted classes:", train_unique_predictions)
print("Validation predicted classes:", val_unique_predictions)

print("Label mapping valid:", mapping_valid)
print("Current optimizer LR:", optimizer.param_groups[0]["lr"])

print("\nDiagnostic completed.")

Running training diagnostics...

=== BERT TRAINING DIAGNOSTIC ===
Training Accuracy: 22.07%
Training Macro F1: 16.72%
Training Loss: 4.0427

Validation Accuracy: 22.70%
Validation Macro F1: 15.74%
Validation Loss: 4.0512

Training predicted classes: 65
Validation predicted classes: 50
Label mapping valid: True
Current optimizer LR: 0.0

Diagnostic completed.


In [24]:
import time
import copy
import torch
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from transformers import BertForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Fresh pretrained model initialization
print("Reloading fresh pretrained BERT-Tiny...")

bert_model_v2 = BertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=77,
    id2label=id2label,
    label2id=label2id,
    problem_type="single_label_classification",
)

bert_model_v2.to(device)

# Improved training configuration
EPOCHS_V2 = 8
LEARNING_RATE_V2 = 5e-5
WEIGHT_DECAY_V2 = 0.01
PATIENCE = 3

optimizer_v2 = torch.optim.AdamW(
    bert_model_v2.parameters(),
    lr=LEARNING_RATE_V2,
    weight_decay=WEIGHT_DECAY_V2
)

total_steps_v2 = EPOCHS_V2 * len(train_loader)
warmup_steps_v2 = int(total_steps_v2 * 0.1)

def lr_lambda_v2(step):
    if step < warmup_steps_v2:
        return float(step) / max(1, warmup_steps_v2)

    return max(
        0.0,
        float(total_steps_v2 - step)
        / max(1, total_steps_v2 - warmup_steps_v2)
    )

scheduler_v2 = torch.optim.lr_scheduler.LambdaLR(
    optimizer_v2,
    lr_lambda_v2
)

# Training history
history_v2 = []
best_macro_f1_v2 = -1.0
best_state_v2 = None
best_epoch_v2 = 0
epochs_without_improvement = 0

print("\n=== BERT-TINY IMPROVED TRAINING ===")
print("Learning rate:", LEARNING_RATE_V2)
print("Maximum epochs:", EPOCHS_V2)
print("Early stopping patience:", PATIENCE)

for epoch in range(EPOCHS_V2):

    epoch_start = time.time()

    # Training
    bert_model_v2.train()
    total_train_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS_V2}"
    )

    for batch in progress:

        input_ids, attention_mask, labels = [
            tensor.to(device) for tensor in batch
        ]

        optimizer_v2.zero_grad(set_to_none=True)

        outputs = bert_model_v2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            bert_model_v2.parameters(),
            max_norm=1.0
        )

        optimizer_v2.step()
        scheduler_v2.step()

        total_train_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_train_loss = total_train_loss / len(train_loader)

    # Validation
    bert_model_v2.eval()

    total_val_loss = 0.0
    all_predictions = []
    all_true_labels = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids, attention_mask, labels = [
                tensor.to(device) for tensor in batch
            ]

            outputs = bert_model_v2(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_val_loss += outputs.loss.item()

            predictions = outputs.logits.argmax(dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    average_val_loss = total_val_loss / len(val_loader)

    val_accuracy = accuracy_score(
        all_true_labels,
        all_predictions
    )

    val_macro_f1 = f1_score(
        all_true_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    val_weighted_f1 = f1_score(
        all_true_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    epoch_time = time.time() - epoch_start

    # Checkpoint selection
    if val_macro_f1 > best_macro_f1_v2:

        best_macro_f1_v2 = val_macro_f1
        best_epoch_v2 = epoch + 1
        epochs_without_improvement = 0

        best_state_v2 = {
            key: value.detach().cpu().clone()
            for key, value in bert_model_v2.state_dict().items()
        }

    else:
        epochs_without_improvement += 1

    history_v2.append({
        "Epoch": epoch + 1,
        "Train Loss": average_train_loss,
        "Validation Loss": average_val_loss,
        "Accuracy": val_accuracy,
        "Macro F1": val_macro_f1,
        "Weighted F1": val_weighted_f1,
        "Time (sec)": epoch_time
    })

    print(f"\nEpoch {epoch + 1}/{EPOCHS_V2}")
    print(f"Train Loss: {average_train_loss:.4f}")
    print(f"Validation Loss: {average_val_loss:.4f}")
    print(f"Accuracy: {val_accuracy:.2%}")
    print(f"Macro F1: {val_macro_f1:.2%}")
    print(f"Weighted F1: {val_weighted_f1:.2%}")
    print(f"Time: {epoch_time:.2f} sec")
    print(f"Best Macro F1 so far: {best_macro_f1_v2:.2%}")

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

# Restore best checkpoint
bert_model_v2.load_state_dict(best_state_v2)
bert_model_v2.to(device)

# Save history
history_v2_df = pd.DataFrame(history_v2)

history_v2_df.to_csv(
    REPORT_DIR / "banking77_bert_tiny_improved_training.csv",
    index=False
)

print("\n=== IMPROVED BERT TRAINING SUMMARY ===")
print("Completed epochs:", len(history_v2))
print("Best epoch:", best_epoch_v2)
print(f"Best Validation Macro F1: {best_macro_f1_v2:.2%}")
print("Best checkpoint restored.")
print("Training history saved.")

Reloading fresh pretrained BERT-Tiny...


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 


=== BERT-TINY IMPROVED TRAINING ===
Learning rate: 5e-05
Maximum epochs: 8
Early stopping patience: 3


Epoch 1/8:   0%|          | 0/563 [00:00<?, ?it/s]


Epoch 1/8
Train Loss: 4.2890
Validation Loss: 4.0543
Accuracy: 21.10%
Macro F1: 14.83%
Weighted F1: 15.59%
Time: 125.38 sec
Best Macro F1 so far: 14.83%


... (10 outputs left out)

Epoch 7/8:   0%|          | 0/563 [00:00<?, ?it/s]


Epoch 7/8
Train Loss: 2.4205
Validation Loss: 2.3075
Accuracy: 59.70%
Macro F1: 51.32%
Weighted F1: 54.87%
Time: 92.55 sec
Best Macro F1 so far: 51.44%


Epoch 8/8:   0%|          | 0/563 [00:00<?, ?it/s]


Epoch 8/8
Train Loss: 2.3620
Validation Loss: 2.2806
Accuracy: 59.50%
Macro F1: 50.98%
Weighted F1: 54.61%
Time: 101.60 sec
Best Macro F1 so far: 51.44%

=== IMPROVED BERT TRAINING SUMMARY ===
Completed epochs: 8
Best epoch: 6
Best Validation Macro F1: 51.44%
Best checkpoint restored.
Training history saved.


In [25]:
print("========== BERT TRAINING COMPLETE ==========")

for _, row in history_v2_df.iterrows():
    print(
        f"Epoch {int(row['Epoch'])}: "
        f"Accuracy = {row['Accuracy']:.2%} | "
        f"Macro F1 = {row['Macro F1']:.2%} | "
        f"Train Loss = {row['Train Loss']:.4f} | "
        f"Val Loss = {row['Validation Loss']:.4f}"
    )

print("\n========== FINAL SUMMARY ==========")
print("Completed Epochs:", len(history_v2_df))
print("Best Epoch:", best_epoch_v2)
print(f"Best Validation Macro F1: {best_macro_f1_v2:.2%}")

print("\n==========================================")

========== BERT TRAINING COMPLETE ==========
Epoch 1: Accuracy = 21.10% | Macro F1 = 14.83% | Train Loss = 4.2890 | Val Loss = 4.0543
Epoch 2: Accuracy = 40.00% | Macro F1 = 31.58% | Train Loss = 3.8205 | Val Loss = 3.4601
Epoch 3: Accuracy = 49.10% | Macro F1 = 40.61% | Train Loss = 3.3225 | Val Loss = 3.0339
Epoch 4: Accuracy = 52.90% | Macro F1 = 44.29% | Train Loss = 2.9598 | Val Loss = 2.7360
Epoch 5: Accuracy = 55.80% | Macro F1 = 47.27% | Train Loss = 2.7080 | Val Loss = 2.5258
Epoch 6: Accuracy = 59.40% | Macro F1 = 51.44% | Train Loss = 2.5309 | Val Loss = 2.3864
Epoch 7: Accuracy = 59.70% | Macro F1 = 51.32% | Train Loss = 2.4205 | Val Loss = 2.3075
Epoch 8: Accuracy = 59.50% | Macro F1 = 50.98% | Train Loss = 2.3620 | Val Loss = 2.2806

========== FINAL SUMMARY ==========
Completed Epochs: 8
Best Epoch: 6
Best Validation Macro F1: 51.44%



In [26]:
import json
from pathlib import Path

# Resolve actual project root
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name.lower() == "notebooks"
    else Path.cwd()
)

BERT_SAVE_DIR = (
    PROJECT_ROOT
    / "models"
    / "intent_classification"
    / "banking77_bert_tiny_best"
)

BERT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save best restored model and tokenizer
bert_model_v2.save_pretrained(BERT_SAVE_DIR)
bert_tokenizer.save_pretrained(BERT_SAVE_DIR)

# Save training history
history_v2_df.to_csv(
    PROJECT_ROOT / "reports" / "models"
    / "banking77_bert_tiny_training_history.csv",
    index=False
)

# Save metadata
bert_metadata = {
    "model": "BERT-Tiny",
    "dataset": "BANKING77",
    "classes": 77,
    "best_epoch": int(best_epoch_v2),
    "validation_macro_f1": float(best_macro_f1_v2),
    "validation_accuracy": float(
        history_v2_df.loc[
            history_v2_df["Epoch"] == best_epoch_v2,
            "Accuracy"
        ].iloc[0]
    ),
    "max_length": MAX_LENGTH,
    "training_epochs": len(history_v2_df),
}

with open(
    BERT_SAVE_DIR / "training_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(bert_metadata, file, indent=4)

print("=== BERT MODEL SAVED ===")
print("Best Epoch:", best_epoch_v2)
print(f"Validation Macro F1: {best_macro_f1_v2:.2%}")
print("Classes:", 77)
print("Model + Tokenizer: Saved")
print("Training History: Saved")
print("Location:", BERT_SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

=== BERT MODEL SAVED ===
Best Epoch: 6
Validation Macro F1: 51.44%
Classes: 77
Model + Tokenizer: Saved
Training History: Saved
Location: d:\Data scientist\AI Customer Support Intelligence\models\intent_classification\banking77_bert_tiny_best


In [27]:
import json
import pandas as pd
import torch

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load official test data
test_df_bert = pd.read_csv(TEST_PATH)

X_test_bert = test_df_bert["text"].astype(str)
y_test_bert = test_df_bert["category"].astype(str)

# Convert labels using existing mapping
y_test_ids = y_test_bert.map(label2id)

assert y_test_ids.notna().all()
assert y_test_bert.nunique() == 77

# Tokenize official test data
test_encodings_bert = bert_tokenizer(
    X_test_bert.tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

test_dataset_bert = TensorDataset(
    test_encodings_bert["input_ids"],
    test_encodings_bert["attention_mask"],
    torch.tensor(y_test_ids.values, dtype=torch.long)
)

test_loader_bert = DataLoader(
    test_dataset_bert,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# Evaluate restored best checkpoint
bert_model_v2.eval()

all_test_predictions = []
all_test_labels = []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader_bert:

        outputs = bert_model_v2(
            input_ids=input_ids.to(device),
            attention_mask=attention_mask.to(device)
        )

        predictions = outputs.logits.argmax(dim=1)

        all_test_predictions.extend(predictions.cpu().numpy())
        all_test_labels.extend(labels.numpy())

# Calculate metrics
test_accuracy_bert = accuracy_score(
    all_test_labels,
    all_test_predictions
)

test_macro_f1_bert = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1_bert = f1_score(
    all_test_labels,
    all_test_predictions,
    average="weighted",
    zero_division=0
)

# Save detailed report
bert_test_report = classification_report(
    all_test_labels,
    all_test_predictions,
    target_names=bert_classes,
    output_dict=True,
    zero_division=0
)

report_dir_final = PROJECT_ROOT / "reports" / "models"
report_dir_final.mkdir(parents=True, exist_ok=True)

pd.DataFrame(bert_test_report).T.to_csv(
    report_dir_final / "banking77_bert_tiny_test_report.csv"
)

# Save metrics
bert_test_metrics = {
    "model": "BERT-Tiny",
    "test_records": len(test_df_bert),
    "training_records": len(X_train_bert),
    "best_epoch": int(best_epoch_v2),
    "test_accuracy": float(test_accuracy_bert),
    "test_macro_f1": float(test_macro_f1_bert),
    "test_weighted_f1": float(test_weighted_f1_bert),
}

with open(
    report_dir_final / "banking77_bert_tiny_test_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(bert_test_metrics, f, indent=4)

# Compact output
print("=== BERT-TINY OFFICIAL TEST RESULTS ===")
print("Test records:", len(test_df_bert))
print(f"Accuracy: {test_accuracy_bert:.2%}")
print(f"Macro F1: {test_macro_f1_bert:.2%}")
print(f"Weighted F1: {test_weighted_f1_bert:.2%}")
print("Best epoch:", best_epoch_v2)
print("Detailed report: Saved")
print("Metrics JSON: Saved")

=== BERT-TINY OFFICIAL TEST RESULTS ===
Test records: 3080
Accuracy: 54.64%
Macro F1: 49.19%
Weighted F1: 49.19%
Best epoch: 6
Detailed report: Saved
Metrics JSON: Saved


In [28]:
import json
import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": "Linear SVM",
        "Type": "Classical ML",
        "Training Records": 9993,
        "Validation Macro F1": 0.8959,
        "Test Accuracy": test_accuracy,
        "Test Macro F1": test_macro_f1,
        "Test Weighted F1": test_weighted_f1,
    },
    {
        "Model": "BERT-Tiny",
        "Type": "Deep Learning",
        "Training Records": 8993,
        "Validation Macro F1": best_macro_f1_v2,
        "Test Accuracy": test_accuracy_bert,
        "Test Macro F1": test_macro_f1_bert,
        "Test Weighted F1": test_weighted_f1_bert,
    }
])

# Select winner using official test results for reporting only
# No further tuning will be performed using test results.
winner = comparison.loc[
    comparison["Validation Macro F1"].idxmax(),
    "Model"
]

REPORT_PATH = (
    PROJECT_ROOT / "reports" / "models"
    / "banking77_ml_vs_dl_comparison.csv"
)

comparison.to_csv(REPORT_PATH, index=False)

summary = {
    "dataset": "BANKING77",
    "selected_model": "Linear SVM",
    "selection_basis": "Validation Macro F1",
    "svm_official_test_accuracy": float(test_accuracy),
    "bert_official_test_accuracy": float(test_accuracy_bert),
    "svm_official_test_macro_f1": float(test_macro_f1),
    "bert_official_test_macro_f1": float(test_macro_f1_bert),
    "comparison_note": (
        "Training record counts differ; SVM used 9993 and "
        "BERT-Tiny used 8993 records."
    )
}

with open(
    PROJECT_ROOT / "reports" / "models"
    / "banking77_model_selection.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4)

print("=== BANKING77 FINAL MODEL COMPARISON ===")

for _, row in comparison.iterrows():
    print(
        f"{row['Model']} | "
        f"Test Accuracy: {row['Test Accuracy']:.2%} | "
        f"Test Macro F1: {row['Test Macro F1']:.2%}"
    )

print("\nSelected Model:", "Linear SVM")
print("Comparison CSV: Saved")
print("Selection JSON: Saved")

=== BANKING77 FINAL MODEL COMPARISON ===
Linear SVM | Test Accuracy: 90.88% | Test Macro F1: 90.88%
BERT-Tiny | Test Accuracy: 54.64% | Test Macro F1: 49.19%

Selected Model: Linear SVM
Comparison CSV: Saved
Selection JSON: Saved


In [29]:
from pathlib import Path
import pandas as pd

# Resolve project root
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name.lower() == "notebooks"
    else Path.cwd()
)

TWITTER_DIR = PROJECT_ROOT / "data" / "processed" / "twitter"

TWITTER_CLEAN_PATH = (
    TWITTER_DIR / "customer_support_twitter_clean.csv"
)

TWITTER_PAIRS_PATH = (
    TWITTER_DIR / "twitter_reply_pairs.csv"
)

# Verify files
files = {
    "Clean Twitter Dataset": TWITTER_CLEAN_PATH,
    "Twitter Reply Pairs": TWITTER_PAIRS_PATH,
}

print("=== TWITTER DATASET VERIFICATION ===")

for name, path in files.items():
    print(
        f"{name}: "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )

# Inspect schema without loading complete files
if TWITTER_CLEAN_PATH.exists():

    sample = pd.read_csv(
        TWITTER_CLEAN_PATH,
        nrows=5
    )

    print("\nTwitter columns:", list(sample.columns))

if TWITTER_PAIRS_PATH.exists():

    pairs_sample = pd.read_csv(
        TWITTER_PAIRS_PATH,
        nrows=5
    )

    print("Reply pair columns:", list(pairs_sample.columns))

print("\nDataset verification completed.")

=== TWITTER DATASET VERIFICATION ===
Clean Twitter Dataset: FOUND
Twitter Reply Pairs: FOUND

Twitter columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']
Reply pair columns: ['parent_tweet_id', 'parent_author_id', 'parent_inbound', 'parent_created_at', 'parent_text', 'reply_tweet_id', 'reply_author_id', 'reply_inbound', 'reply_created_at', 'reply_text', 'pair_type']

Dataset verification completed.
